# V8 · Baseline: CatBoost with native categorical handling

**Held-out test RMSLE: 0.20456** · 5-fold OOF RMSLE: 0.20843

- CatBoost (5,000 iterations, lr 0.03, depth 8) using its built-in ordered target statistics on **42 raw categorical columns**
- Features: transaction year, month and quarter; `AssetAge`; hour-meter log and missing flag; `"Unknown"` / `"Not Applicable"` fills
- Very slow: about **58 minutes per fold** on CPU because of the high-cardinality categoricals

**Takeaway:** a strong zero-encoding baseline, but slow to train. Explicit encodings with XGBoost (V9) scored better and ran much faster.

> This is the original notebook as run on Kaggle. Data paths point to `/kaggle/input/...`; to run it locally, change them to your data folder. Code and outputs are unchanged.

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/sample_submission.csv
/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/train.csv
/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/metadata.csv
/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/test.csv


In [2]:
trainData = pd.read_csv("/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/train.csv")
testData = pd.read_csv("/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/test.csv")


/tmp/ipykernel_16/681260659.py:1: DtypeWarning: Columns (38,39) have mixed types. Specify dtype option on import or set low_memory=False.
  trainData = pd.read_csv("/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/train.csv")


In [3]:
trainData.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 138701 entries, 0 to 138700
Data columns (total 50 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   TransactionID              138701 non-null  int64  
 1   TargetValue                138701 non-null  float64
 2   AssetID                    138701 non-null  int64  
 3   ProductConfigID            138701 non-null  int64  
 4   DataOriginCode             138701 non-null  object 
 5   VendorPartnerID            138701 non-null  object 
 6   ManufactureYear            138701 non-null  int64  
 7   OperationalHoursMeter      82058 non-null   float64
 8   UtilizationTier            50475 non-null   object 
 9   TransactionDate            138701 non-null  object 
 10  Spec_FullDescriptor        138701 non-null  object 
 11  Spec_BaseClass             138701 non-null  object 
 12  Spec_SubClass              101716 non-null  object 
 13  Spec_ReleaseSeries         32

In [4]:
trainData.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
TransactionID,138701.0,NaN,NaN,NaN,2186275.950556,1128496.619347,1139309.0,1465754.0,1751884.0,2372139.0,6333410.0
TargetValue,138701.0,NaN,NaN,NaN,41521.874047,26361.125948,7500.0,20500.0,35000.0,57000.0,142000.0
AssetID,138701.0,NaN,NaN,NaN,1204814.320423,528737.593449,8.0,1007930.0,1246956.0,1495817.0,2478476.0
ProductConfigID,138701.0,NaN,NaN,NaN,7286.421922,7542.421537,39.0,1693.0,3852.0,11873.0,37209.0
DataOriginCode,138701,6,ADI149,59541,NaN,NaN,NaN,NaN,NaN,NaN,NaN
VendorPartnerID,138701,31,UJF,67828,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ManufactureYear,138701.0,NaN,NaN,NaN,1891.640853,306.992001,1001.0,1990.0,1998.0,2004.0,2015.0
OperationalHoursMeter,82058.0,NaN,NaN,NaN,4553.718943,27922.04485,0.0,0.0,1620.0,5001.0,1729600.0
UtilizationTier,50475,3,Medium,24588,NaN,NaN,NaN,NaN,NaN,NaN,NaN
TransactionDate,138701,3676,2010-02-16,1159,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
trainData.head()

,TransactionID,TargetValue,AssetID,ProductConfigID,DataOriginCode,VendorPartnerID,ManufactureYear,OperationalHoursMeter,UtilizationTier,TransactionDate,...,col20,col21,col22,col23,col24,col25,col27,col28,col29,col30
0,1139309,57000.0,117665,88,ACH138,GTX,1997,4640.0,Low,2005-03-26,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Standard,Conventional
1,1139316,26500.0,1001282,4616,ACH138,GTX,2005,508.0,Low,2009-12-18,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1139317,21000.0,772709,1948,ACH138,GTX,1994,11540.0,High,2005-08-26,...,Steel,None or Unspecified,None or Unspecified,None or Unspecified,None or Unspecified,Double,NaN,NaN,NaN,NaN
3,1139322,27000.0,902010,3550,ACH138,GTX,2002,4883.0,High,2006-11-17,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1139333,21500.0,1036259,36014,ACH138,GTX,2009,302.0,Low,2010-08-27,...,Rubber,None or Unspecified,None or Unspecified,None or Unspecified,None or Unspecified,Double,NaN,NaN,NaN,NaN


In [6]:
trainData.nunique()

TransactionID                138701
TargetValue                     675
AssetID                      120111
ProductConfigID                3594
DataOriginCode                    6
VendorPartnerID                  31
ManufactureYear                  68
OperationalHoursMeter         14643
UtilizationTier                   3
TransactionDate                3676
Spec_FullDescriptor            3505
Spec_BaseClass                 1249
Spec_SubClass                   147
Spec_ReleaseSeries              113
Spec_VariantModifier            132
AssetScaleFactor                  6
FunctionalClassification         65
RegionCode                       53
InventoryGroupCategory            6
InventoryGroupDescription         6
col1                              4
CabinType                         3
Forks                             2
col3                              3
col4                              2
DrivetrainType                    8
col5                              2
col6                        

In [7]:
trainData.isna().mean()*100

TransactionID                 0.000000
TargetValue                   0.000000
AssetID                       0.000000
ProductConfigID               0.000000
DataOriginCode                0.000000
VendorPartnerID               0.000000
ManufactureYear               0.000000
OperationalHoursMeter        40.838206
UtilizationTier              63.608770
TransactionDate               0.000000
Spec_FullDescriptor           0.000000
Spec_BaseClass                0.000000
Spec_SubClass                26.665273
Spec_ReleaseSeries           76.282795
Spec_VariantModifier         62.269919
AssetScaleFactor             29.711394
FunctionalClassification      0.000000
RegionCode                    0.000000
InventoryGroupCategory        0.000000
InventoryGroupDescription     0.000000
col1                         75.641127
CabinType                     0.000000
Forks                        78.861724
col3                         78.909309
col4                         92.515555
DrivetrainType           

In [8]:
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_log_error

from catboost import CatBoostRegressor

In [9]:
train_ids = trainData["TransactionID"].copy()
test_ids = testData["TransactionID"].copy()

In [10]:
trainData = trainData.drop(columns=["TransactionID", "InventoryGroupCategory"], errors="ignore")

In [11]:
trainData["TransactionDate"] = pd.to_datetime(trainData["TransactionDate"])
trainData["TransactionYear"] = trainData["TransactionDate"].dt.year
trainData["TransactionMonth"] = trainData["TransactionDate"].dt.month
trainData["TransactionQuarter"] = trainData["TransactionDate"].dt.quarter
trainData = trainData.drop(columns="TransactionDate")

In [12]:
trainData["ManufactureYearMissing"] = (trainData["ManufactureYear"] == 1001).astype(int)
trainData["ManufactureYear"] = trainData["ManufactureYear"].replace(1001, np.nan)
trainData["AssetAge"] = trainData["TransactionYear"] - trainData["ManufactureYear"]
trainData.loc[trainData["AssetAge"] < 0, "AssetAge"] = np.nan

In [13]:
trainData["HoursMissing"] = trainData["OperationalHoursMeter"].isna().astype(int)
trainData["OperationalHoursLog"] = np.log1p(trainData["OperationalHoursMeter"])

In [14]:
unknown_cols = ["UtilizationTier", "Spec_SubClass", "Spec_ReleaseSeries", "Spec_VariantModifier", "AssetScaleFactor", "DrivetrainType", "Forks"]
for col in unknown_cols:
    if col in trainData.columns:
        trainData[col] = trainData[col].fillna("Unknown")

In [15]:
machine_cols = ["col1","col3","col4","col5","col6","col7","col8","col9","col10","col11","col12","col13","col14","col15","col16","col18","col19","col20","col21","col22","col23","col24","col25","col27","col28","col29","col30"]
for col in machine_cols:
    if col in trainData.columns:
        trainData[col] = trainData[col].fillna("Not Applicable")

In [16]:
testData = testData.drop(columns=["TransactionID", "InventoryGroupCategory"], errors="ignore")

testData["TransactionDate"] = pd.to_datetime(testData["TransactionDate"])
testData["TransactionYear"] = testData["TransactionDate"].dt.year
testData["TransactionMonth"] = testData["TransactionDate"].dt.month
testData["TransactionQuarter"] = testData["TransactionDate"].dt.quarter
testData = testData.drop(columns="TransactionDate")

testData["ManufactureYearMissing"] = (testData["ManufactureYear"] == 1001).astype(int)
testData["ManufactureYear"] = testData["ManufactureYear"].replace(1001, np.nan)
testData["AssetAge"] = testData["TransactionYear"] - testData["ManufactureYear"]
testData.loc[testData["AssetAge"] < 0, "AssetAge"] = np.nan

testData["HoursMissing"] = testData["OperationalHoursMeter"].isna().astype(int)
testData["OperationalHoursLog"] = np.log1p(testData["OperationalHoursMeter"])

for col in unknown_cols:
    if col in testData.columns:
        testData[col] = testData[col].fillna("Unknown")

for col in machine_cols:
    if col in testData.columns:
        testData[col] = testData[col].fillna("Not Applicable")

In [17]:
print(trainData.shape)
print(testData.shape)

(138701, 54)
(15000, 53)


In [18]:
TARGET = "TargetValue"

y = trainData[TARGET]
X = trainData.drop(columns=TARGET)
X_test = testData.copy()

cat_features = list(X.select_dtypes(include="object").columns)

print("Training Shape :", X.shape)
print("Testing Shape  :", X_test.shape)
print("Categorical Features :", len(cat_features))

Training Shape : (138701, 53)
Testing Shape  : (15000, 53)
Categorical Features : 42


In [19]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

oof_predictions = np.zeros(len(X))
test_predictions = np.zeros(len(X_test))
cv_scores = []

In [20]:
fold_number = 1

for train_index, valid_index in kf.split(X):
    print("\nFold", fold_number)

    X_train = X.iloc[train_index]
    X_valid = X.iloc[valid_index]

    y_train = np.log1p(y.iloc[train_index])
    y_valid = np.log1p(y.iloc[valid_index])

    model = CatBoostRegressor(
        iterations=5000,
        learning_rate=0.03,
        depth=8,
        loss_function="RMSE",
        random_seed=42,
        early_stopping_rounds=200,
        verbose=500
    )

    model.fit(
        X_train,
        y_train,
        cat_features=cat_features,
        eval_set=(X_valid, y_valid),
        use_best_model=True
    )

    # Validation predictions
    valid_pred = model.predict(X_valid)
    valid_pred = np.expm1(valid_pred)
    valid_pred = np.maximum(valid_pred, 0)

    # Test predictions
    test_pred = model.predict(X_test)
    test_pred = np.expm1(test_pred)
    test_pred = np.maximum(test_pred, 0)

    
    y_true_valid = y.iloc[valid_index]
    score = np.sqrt(mean_squared_log_error(y_true_valid, valid_pred))
    cv_scores.append(score)

    # Store predictions
    oof_predictions[valid_index] = valid_pred
    test_predictions = test_predictions + (test_pred / 5)

    print("Fold RMSLE :", round(score, 5))
    
    fold_number = fold_number + 1


Fold 1
0:	learn: 0.6578206	test: 0.6559544	best: 0.6559544 (0)	total: 823ms	remaining: 1h 8m 35s
500:	learn: 0.2328040	test: 0.2296272	best: 0.2296272 (500)	total: 5m 38s	remaining: 50m 40s
1000:	learn: 0.2174234	test: 0.2188080	best: 0.2188080 (1000)	total: 11m 34s	remaining: 46m 15s
1500:	learn: 0.2099038	test: 0.2144821	best: 0.2144821 (1500)	total: 17m 27s	remaining: 40m 41s
2000:	learn: 0.2046668	test: 0.2121424	best: 0.2121424 (2000)	total: 23m 17s	remaining: 34m 54s
2500:	learn: 0.2004013	test: 0.2104877	best: 0.2104877 (2500)	total: 29m 11s	remaining: 29m 9s
3000:	learn: 0.1966133	test: 0.2091229	best: 0.2091229 (3000)	total: 35m 6s	remaining: 23m 23s
3500:	learn: 0.1933233	test: 0.2080805	best: 0.2080805 (3500)	total: 40m 55s	remaining: 17m 31s
4000:	learn: 0.1903216	test: 0.2072812	best: 0.2072812 (4000)	total: 46m 46s	remaining: 11m 40s
4500:	learn: 0.1873186	test: 0.2065236	best: 0.2065236 (4500)	total: 52m 40s	remaining: 5m 50s
4999:	learn: 0.1845472	test: 0.2059368	best:

In [21]:
print("\nCross Validation Results")

for i in range(len(cv_scores)):
    print("Fold " + str(i+1) + ": " + str(round(cv_scores[i], 5)))

mean_score = sum(cv_scores) / len(cv_scores)
print("\nMean RMSLE :", round(mean_score, 5))


Cross Validation Results
Fold 1: 0.20594
Fold 2: 0.20893
Fold 3: 0.20927
Fold 4: 0.20974
Fold 5: 0.20827

Mean RMSLE : 0.20843


In [22]:
oof_preds_safe = np.maximum(oof_predictions, 0)
final_oof = np.sqrt(mean_squared_log_error(y, oof_preds_safe))
print("OOF RMSLE  :", round(final_oof, 5))

OOF RMSLE  : 0.20843


In [23]:
submission = pd.DataFrame()
submission["TransactionID"] = test_ids
submission["TargetValue"] = test_predictions

submission.to_csv("submission.csv", index=False)
print(submission.head())

   TransactionID   TargetValue
0        1139307  72014.036791
1        1139419  78116.657017
2        1139482  28763.740494
3        1139522  20064.178729
4        1139684  13479.451507
